# W05B · A · CLIP을 해부하고 의미로 검색하기
2026-09-30 · 생성형 인공지능 · 주교재 3장, 4.1–4.2

CLIP을 처음 배우는 학생을 위한 실습입니다. 사진 40장은 패키지에 포함되며, 최초 CLIP 가중치 약 605 MB와 의존성을 다운로드합니다. 지난 노트북을 실행하지 않아도 새 사본에서 시작할 수 있습니다.
**CPU 기본 / Python 3.12·3.13 / 패키지 0.1.8 / 고정 태그 2026-fall-w05b**. 유료 API 키와 개인 파일 업로드가 필요하지 않습니다.
Drive에 사본을 저장한 뒤 첫 셀부터 실행하세요. 빈칸은 독립된 연습 변수이므로 None을 남겨도 기본 관찰은 진행됩니다.
출력이 예상과 다르면 오류를 숨기지 말고 마지막으로 성공한 셀부터 확인하세요.

[강의 A](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/course/notion/w05b/w05b-clip-search.md) · [강의 B](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/course/notion/w05b/w05b-diffusion.md) · [기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/course/notion/w05b/w05b-workbook.md) · [API 정의와 사용법](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/W05B-API.md)


## 0. 독립 실행 환경
설치 셀은 필요한 버전을 확인합니다. 이미 import한 패키지를 교체해야 하는 경우 런타임을 재시작하고 첫 셀부터 실행하세요.
설치·모델 준비·실제 계산의 시간을 구분합니다. 모델 준비 실패를 가짜 결과로 바꾸지 않습니다.


In [ ]:
import sys,subprocess,os,time
from pathlib import Path
from importlib.metadata import version,PackageNotFoundError
PUBLIC_REF='2026-fall-w05b'
try:
    ready=all(version(p)==v for p,v in [('luna-genai','0.1.8'),('transformers','5.15.1'),('gradio','6.26.0'),('diffusers','0.40.0')])
except PackageNotFoundError:
    ready=False
if not ready:
    if any(p in sys.modules for p in ('luna_genai','torch','transformers','gradio','diffusers')):
        raise RuntimeError('런타임을 재시작한 뒤 첫 설치 셀부터 실행하세요.')
    package=str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
print('Python',sys.version.split()[0],'; package',version('luna-genai'),'; diffusers',version('diffusers'))
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
torch.set_num_threads(2)
started=time.perf_counter()


## 1. 사진을 먼저 보고, 검색의 정답 기준 세우기
[`CLIPSearch`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/semantic_search.py#L65)는 고정된 B/32 사전학습 모델과 검증된 사진 40장을 준비합니다. 생성자에서 캐시가 없으면 모델을 받으며 학습하지 않습니다.
`gallery`에는 id, 실제 파일 path, 출처 source, 저자 author, 이용 조건 license가 있습니다. 파일명과 category는 출력용이며 이미지 인코더 입력에 넣지 않습니다.
[`CLIPSearch.index_images`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/semantic_search.py#L137)는 사진의 **픽셀**에서 `(40,512)` 임베딩을 계산하거나 일치하는 캐시를 읽습니다.

관찰 순서: 사진 전체 보기 → 관련 사진 ID 정하기 → 검색 실행 → 순위와 비교. 결과를 본 뒤 정답 기준을 바꾸지 않습니다.


In [ ]:
from PIL import Image
from luna_genai.semantic_search import CLIPSearch,unit_rows,precision_at_k
from luna_genai.clip_teaching import contrastive_table,compare_search
from luna_genai.diffusion_app import build_study_app,APP_CSS
lab=CLIPSearch()
vectors=lab.index_images()
print('이미지 행렬:',vectors.shape,'새 모델 파일:',lab.model_downloads or '없음: 캐시 재사용')
print('임베딩 캐시 재사용:',lab.cache_reused)
fig,axes=plt.subplots(5,8,figsize=(14,9))
for ax,row in zip(axes.flat,lab.gallery.itertuples()):
    with Image.open(row.path) as pic: ax.imshow(pic)
    ax.set_title(row.id);ax.axis('off')
plt.tight_layout();plt.show()
display(lab.gallery[['id','source','author','license','license_url']])


### A1 · 검색 전에 관련성 정의하기
“a cat in the snow”와 “a horse on a beach”에 맞는 사진 ID 집합을 정하세요. 대상만 맞으면 되는지, 배경도 맞아야 하는지 기준을 적습니다. 힌트: 위 전체 사진을 확대하세요.


In [ ]:
my_relevant_snow=None  # 예: {'Pxx', 'Pyy'}; 관찰 후 기록
my_relevant_horse=None


## 2. 인코더 내부 표현 → 투영 → 정규화
이미지의 픽셀과 문장의 토큰 ID를 직접 내적할 수 없습니다. 각각 인코더를 거친 뒤 **학습된 투영**으로 같은 차원에 놓습니다.
[`unit_rows`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/semantic_search.py#L23)는 유한하고 0이 아닌 `(N,d)` 텐서의 **각 행**을 L2 길이로 나눕니다. 길이가 0인 벡터는 오류로 보고합니다.
[`CLIPSearch.encode_texts`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/semantic_search.py#L125)는 짧은 문장 목록에서 정규화된 `(N,512)` 벡터를 반환합니다. 최대 77토큰 제한이 있으므로 짧고 구체적인 영어 문장을 기본으로 씁니다.

아래에서는 wrapper를 잠시 열어 실제 내부 크기를 봅니다. [CLIPModel 공식 문서](https://huggingface.co/docs/transformers/model_doc/clip)에서 vision_model, text_model과 projection의 역할을 비교하세요.


In [ ]:
photos=[Image.open(p).convert('RGB') for p in lab.gallery.iloc[[0,8,16]].path]
captions=['a photo of a dog','a photo of a cat','a photo of a pizza']
inputs=lab.processor(images=photos,text=captions,padding=True,return_tensors='pt')
with torch.inference_mode():
    image_hidden=lab.model.vision_model(pixel_values=inputs['pixel_values']).pooler_output
    text_hidden=lab.model.text_model(input_ids=inputs['input_ids'],attention_mask=inputs['attention_mask']).pooler_output
    image_projected=lab.model.visual_projection(image_hidden)
    text_projected=lab.model.text_projection(text_hidden)
    image_unit=unit_rows(image_projected)
    text_unit=unit_rows(text_projected)
display(pd.DataFrame([
    ['pixel_values',tuple(inputs['pixel_values'].shape)],['input_ids',tuple(inputs['input_ids'].shape)],
    ['image hidden',tuple(image_hidden.shape)],['text hidden',tuple(text_hidden.shape)],
    ['image projected',tuple(image_projected.shape)],['text projected',tuple(text_projected.shape)],
    ['image normalized',tuple(image_unit.shape)],['text normalized',tuple(text_unit.shape)]],columns=['단계','shape']))
print('행별 길이:',image_unit.norm(dim=1),text_unit.norm(dim=1))


### A2 · 차원과 정규화 축
왜 이미지 768차원과 문장 512차원을 그대로 내적할 수 없나요? 40×512 행렬에서 행별 정규화의 dim은 무엇인가요? 힌트: 한 사진은 한 행입니다.


### 작은 수계산과 실제 전처리
설명용 두 벡터는 $(3,4)$와 $(0,2)$입니다. 길이로 나눈 뒤 내적하면 $0.8$입니다. 실제 512차원에서도 같은 연산을 합니다.
전처리는 resize·center crop·픽셀 정규화를 수행합니다. 아래 오른쪽 그림은 모델에 실제 들어간 픽셀을 시각화하기 위해 정규화만 되돌린 것입니다. 가장자리가 잘리면 작은 대상이나 관계 단서가 사라질 수 있습니다.


In [ ]:
toy=torch.tensor([[3.,4.],[0.,2.]])
print('원래 내적:',float(toy[0]@toy[1]))
print('벡터 길이:',toy.norm(dim=1).tolist())
print('정규화 뒤 내적:',float(unit_rows(toy)[0]@unit_rows(toy)[1]))
pixel=inputs['pixel_values'][0].clone()
mean=torch.tensor(lab.processor.image_processor.image_mean)[:,None,None]
std=torch.tensor(lab.processor.image_processor.image_std)[:,None,None]
fig,axes=plt.subplots(1,2,figsize=(8,4))
axes[0].imshow(photos[0]);axes[0].set_title('Original photo')
axes[1].imshow((pixel*std+mean).permute(1,2,0).clamp(0,1));axes[1].set_title('Actual model crop: 224 x 224')
for ax in axes:ax.axis('off')
plt.show()


## 3. CLIP 학습의 짝맞추기: 3×3 행렬
[`contrastive_table`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/clip_teaching.py#L9)는 같은 순서의 `(N,d)` 이미지·문장 벡터를 받아 코사인, logits, 양방향 softmax, 평균 대조 손실을 반환합니다. 학습은 수행하지 않습니다.
scale 기본값 10은 작은 설명용 설정입니다. 아래에는 실제 모델의 `exp(logit_scale)`을 넣습니다.

$$S_{ij}=\frac{u_i^\top v_j}{\lVert u_i\rVert\lVert v_j\rVert},\qquad L=\frac{\mathrm{CE}(aS, y)+\mathrm{CE}(aS^\top,y)}{2}.$$

대각선은 함께 제공한 이미지–문장 쌍입니다. 이미지 하나에서 문장 후보를 고르는 방향과 문장 하나에서 이미지 후보를 고르는 방향을 모두 학습합니다. 이 작은 표는 사전학습 모델의 추론값을 관찰한 것이며 이번 수업에서 CLIP을 재학습하지 않습니다.


In [ ]:
paired=contrastive_table(image_projected,text_projected,scale=float(lab.model.logit_scale.exp().detach()))
for key in ['cosine','image_to_text','text_to_image']:
    print(key);display(pd.DataFrame(paired[key].detach().numpy(),index=['dog','cat','pizza'],columns=['dog','cat','pizza']))
print('대조 손실:',float(paired['loss']))


### A3 · 행·열과 손실 해석
이미지 하나의 올바른 문장이 높은 점수를 받으면 행 손실은 어떻게 되나요? text_to_image에서 행과 열은 각각 무엇인가요? 힌트: 전치한 뒤 행 softmax를 합니다.


## 4. 후보가 바뀌면 상대 점수가 바뀐다
아래는 모델 측정값이 아닌 설명용 점수입니다. A의 cosine 0.8은 유지하고 C를 추가합니다. [torch.softmax](https://docs.pytorch.org/docs/2.8/generated/torch.nn.functional.softmax.html)는 선택한 축의 합이 1이 되게 변환합니다. 후보 전체에 정답이 없어도 합은 1입니다.


In [ ]:
two_scores=torch.tensor([.8,.2])
three_scores=torch.tensor([.8,.2,.7])
print('2개 후보 상대 점수:',torch.softmax(two_scores*5,dim=0).tolist())
print('3개 후보 상대 점수:',torch.softmax(three_scores*5,dim=0).tolist())


### A4 · 점수를 확률로 잘못 해석한 문장 고치기
“A의 softmax가 0.95이므로 이 사진이 정답일 확률은 95%다”를 어떻게 수정하겠습니까? 힌트: 후보 수와 scale.


## 5. 검색을 행렬곱과 정렬로 검산하기
[`CLIPSearch.search`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/semantic_search.py#L159)는 준비된 인덱스에 질의를 넣어 rank·id·cosine·출처 표를 돌려줍니다. 기본 k=3, 같은 점수는 사진 순서를 유지합니다.
이미지 임베딩은 한 번 준비하고, 질의가 바뀔 때 문장만 인코딩합니다. 인덱싱과 질의 처리를 분리하면 무엇을 재사용하는지 보입니다.


In [ ]:
query='a cat in the snow'
query_vector=lab.encode_texts([query])[0]
scores=vectors@query_vector
manual_order=np.argsort(-scores.numpy(),kind='stable')[:3]
manual_ids=lab.gallery.iloc[manual_order].id.tolist()
search_result=lab.search(query,3)
print('query shape:',query_vector.shape,'전체 점수 shape:',scores.shape,'직접 정렬 ID:',manual_ids)
display(search_result[['rank','id','cosine','source']])


### A5 · 검색 빈칸
정규화한 이미지 행렬 E와 질의 q로 전체 점수를 계산한 다음 큰 값부터 3개 ID를 얻는 코드를 작성하세요. 힌트: @, 음수 정렬. 자가 점검: wrapper 검색과 순서가 같아야 합니다.


In [ ]:
my_scores=None  # vectors와 query_vector로 계산
my_top_ids=None


### 대상·속성·관계·부재를 구분해 관찰하기
[`precision_at_k`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/semantic_search.py#L53)는 첫 k개 중 관련 ID 수를 k로 나눕니다. 기본 k=3이며 같은 ID 중복은 거절합니다.
눈 위 고양이 관련 ID는 P10·P11, 고양이와 도마뱀은 P16, 말은 없습니다. 관련 사진이 1장뿐이면 P@3 최댓값은 1/3입니다. 적은 후보 모음의 한계와 모델 오류를 구분하세요.
속성 비교는 “a cat in the snow”와 “a cat indoors”를 같은 k로 비교합니다. 관계 질의도 두 대상이 함께 있는지 사진을 직접 확인합니다.


In [ ]:
queries=[('a cat in the snow',{'P10','P11'}),('a cat and a lizard',{'P16'}),('a horse on a beach',set())]
rows=[]
for q,relevant in queries:
    result=lab.search(q,3)
    rows.append(dict(query=q,ids=result.id.tolist(),precision_at_3=precision_at_k(result.id.tolist(),relevant,3),maximum_possible=min(len(relevant),3)/3))
display(pd.DataFrame(rows))
comparison=compare_search('a cat in the snow','a cat indoors',3,lab)
display(comparison[2]);print(comparison[3])


### A6 · 없는 대상과 평가 분모
말 검색에서 결과가 반환되면 성공인가요? 관련 사진이 2장뿐인 눈 위 고양이 검색의 P@3 상한은 얼마인가요? 한국어 질의 한 번으로 다국어 능력을 일반화할 수 있나요?


## 6. 웹 앱으로 두 검색을 비교하기
[`compare_search`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/clip_teaching.py#L28)의 입력은 문장 A, 문장 B, k, 준비된 lab입니다. 두 gallery·합친 점수표·설명 문자열을 반환하며 모델을 재학습하지 않습니다.
[`build_study_app`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/diffusion_app.py#L57)는 준비된 객체로 화면과 버튼을 만들며 서버는 시작하지 않습니다. [`build_study_app`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05b/src/luna_genai/diffusion_app.py#L57)에 search_lab만 주면 검색 탭이 만들어집니다.
Gradio [이벤트 연결 안내](https://www.gradio.app/guides/blocks-and-event-listeners)를 참고하여 입력 순서와 출력 순서를 대응시키세요.


### A7 · 버튼 callback 조립
Textbox 두 개와 Slider 하나를 compare_search에 연결하는 click 코드를 작성하세요. 출력은 Gallery 두 개, Dataframe, Markdown입니다. 힌트: partial로 lab을 고정합니다.


In [ ]:
my_callback=None  # 연습 callback; 기본 앱은 아래에서 독립적으로 실행됩니다.
app=build_study_app(search_lab=lab)
test_view=compare_search('a cat in the snow','a cat indoors',3,lab)
print('앱 callback 확인:',len(test_view[0]),len(test_view[1]),test_view[2].shape)


In [ ]:
assert vectors.shape==(40,512) and torch.isfinite(vectors).all()
assert torch.allclose(vectors.norm(dim=1),torch.ones(40),atol=1e-5)
assert manual_ids==search_result.id.tolist()
assert len(test_view[0])==len(test_view[1])==3 and len(test_view[2])==6
assert rows[-1]['precision_at_3']==0
print('W05B A 검증 통과: 실제 CLIP, 차원, 정렬, 평가, 앱 callback')
print('환경 설치 이후 경과:',round(time.perf_counter()-started,1),'초')


### 마지막: 앱을 새 탭에서 열기
아래 셀의 공개 공유 링크를 눌러 “두 검색 결과 비교” 버튼을 직접 실행하세요. 이 링크는 Colab 런타임이 살아 있을 때만 유지됩니다.
사진은 검증된 40개 파일 경로만 제공하도록 허용합니다. 텍스트 결과만 보이고 사진이 비어 있으면 새 런타임의 첫 셀부터 실행하여 패키지와 앱 버전을 맞춥니다.
앱에서 문장 하나만 바꾸고 결과 ID·점수·실제 장면을 기록하세요. 실습 B는 별도의 새 사본에서 이어집니다.


In [ ]:
if IN_COLAB:
    app.launch(share=True,inline=False,allowed_paths=lab.gallery.path.tolist(),css=APP_CSS,show_error=True)
    print('앱 새 탭:',app.share_url)
else:
    print('로컬 실행: app.launch(inbrowser=True, allowed_paths=lab.gallery.path.tolist(), css=APP_CSS)')
